# Scope

Scope in python (or any programming language) simply refers to where a particular function or variable is accessible

So basically, where in the program can this function or variable be called and used.

This used to be a very simple concept

where variables inside functions belong to the function, variables in the program belong to the program

But as we divded deeper into first class functions and higher order functions... this becomes increasingly confusing

There are 4 different types of scopes which we care about:

- built-in
- local
- global
- non-local

We care about them so we can avoid naming conflicts and ensuring proper variable usage

## Built In Scope

As the name suggests, the built in scope refers to all the possible things directly accessible whenever we open a python file

Like some data types (list, dict, set, tuple...)

Or some functions (print(), range(), type()...)

Or exceptions ('TypeError', 'ValueError', 'ZeroDivisionError', 'IndexError', 'KeyError')

We don't have to make any imports, and can directly use these things.

In [ ]:
print("You can use print() directly without calling it!")

## Global Scope

Any variable that gets declared on the first indentation is a global variable (basically without any tabs)

You can call it wherever you'd like to call it, including inside functions and classes.

In [1]:
x = 1
print(x)

1


## Local Scope

ok... now is where things start to become fun, local scope is the scope that contains variables defined within a function or a class method

these variables cannot be accessed from outside the function or class that defined it

In [2]:
def foo():
    y = 10

foo()
print(y)

NameError: name 'y' is not defined

This error occurs because we are trying to access a variable from outside it's defined function

Now what happens when we start trying to alter a global variable from a local scope?

In [5]:
x = 1

def foo():
    x = 11
    y = 12
    print(f"Local scope sees x as {x}")
    print(f"Local scope sees y as {y}")

foo()
print(f"Gloabl scope sees x as {x}")

Local scope sees x as 11
Local scope sees y as 12
Gloabl scope sees x as 1


So when we assigned x to 10 inside the function, the local scope does see x as 10

but once the function finished and we see x from the gloabl scope, it's still 1, it did not get affected

So is there a way to change the variables in the global scope from a local scope?

Yeah, it's called calling "global" inside a local function

It basically tells python whenver you use the variable "x" in the local scope, it refers to the x in the global scope

In [6]:
x = 1

def foo():
    global x
    x = 11
    y = 12
    print(f"Local scope sees x as {x}")
    print(f"Local scope sees y as {y}")


foo()
print(f"Gloabl scope sees x as {x}")

Local scope sees x as 11
Local scope sees y as 12
Gloabl scope sees x as 11


Double sided sword, it can help you define constants that can be called anywhere inside functions

But sometimes it can cause trouble when we have functions that alter the global variable

which changes behavior of other functiosn that need the constant

## Non-local scope

this is where things truly start to become weird...  non-local scope refers to the scope that is used to access variables from an enclosing (but non-global) scope within a nested function

When a variable is defined in an outer function and accessed or modified within an inner function, it is considered nonlocal to the inner function

In [7]:
x = 1

def outer():
    x = 2
    def inner():
        x = 3
        print(f"inner function sees x as {x}")
    inner()
    print(f"outer function sees x as {x}")

outer()
print(f"global sees x as {x}")

inner function sees x as 3
outer function sees x as 2
global sees x as 1


Ok... what is happening here?

We call outer, which sets it's local x to 2

Then it defines a function, inner, which set's it's local x to 3

we then print the local x of the inner function, 3

then we priint the local x of the outer function, 2

The function is complete, then we go to global scope and the global x is still 1

Now, what happens when inner doesn't define x = 3?

In [8]:
x = 1

def outer():
    x = 2
    def inner():
        print(f"inner function sees x as {x}")
    inner()
    print(f"outer function sees x as {x}")

outer()
print(f"global sees x as {x}")

inner function sees x as 2
outer function sees x as 2
global sees x as 1


inner would attempt to find the first x defined closest to it's scope

it searches through it's own scope, and finds that x isn't defined

so it would then search 1 scope higher for the existance of a variable called "x"

and it finds that x is defined is as 2 from the scope of outer

now, what if we remove x = 2 from the scope of outer?

In [9]:
x = 1

def outer():
    def inner():
        print(f"inner function sees x as {x}")
    inner()
    print(f"outer function sees x as {x}")

outer()
print(f"global sees x as {x}")

inner function sees x as 1
outer function sees x as 1
global sees x as 1


every function sees x as 1!

as inner searches up and up, it reaches global and finds x as 1, outer also searches up and sees x as 1

and now finally I'll introduce non-local, we know if a function wants to change a global variable, it only needs to call the keyworrd "global" to have the ability to change it

"nonlocal" means to access the variable 1 scope above it

like in this case if I add "nonlocal x" to inner, then inner will refer to the x 1 scope above it, which is the x defined in outer

In [10]:
x = 1

def outer():
    x = 2
    def inner():
        nonlocal x
        x = 3
        print(f"inner function sees x as {x}")
    inner()
    print(f"outer function sees x as {x}")

outer()
print(f"global sees x as {x}")

inner function sees x as 3
outer function sees x as 3
global sees x as 1


The logic makes sense! when we run outer, it first sets it's local x to 2

Then it defines and runs inner, inner has "nonlocal x", so when it sets it's local x to 3, it also changed the local x in the scope above it, outer from 2 to 3.

When outer then prints what it sees x as, it's been changed to 3 by inner, so it sees x as 3

# Interesting Side Effect

What do you think will this function output?

In [ ]:
x = 1

def func():
    print(x)
    x = 2

func()

Logically... it feels like it should first see that there's no local x defined when it runs the print statement, so it should refer to the global x, and print 1... right?

In [11]:
x = 1

def func():
    print(x)
    x = 2

func()

UnboundLocalError: cannot access local variable 'x' where it is not associated with a value

Huh? It's actually an error that comes in compile time.

You can think it this way... the print function doesn't see a local x yet, so it searches for a global one

But then, we tell it that there is a local x variable? 

The function already refers to x as a variable above it's scope, then this definition would conflict, so a error is raised